# Experimento — Perceptron OR

**Aula 1 · 11/11 · Inteligência Artificial aplicada à Engenharia Elétrica**

## Antes de começar

### Objetivo
Classificar dois indicadores de um motor como alerta (OR), a partir de entradas binárias sem unidade (0 ou 1). Pesos e viés são adimensionais.

### Pré-requisitos
Nenhum notebook anterior. Execute este arquivo de forma independente em uma sessão nova do Google Colab.

### Como executar
1. Abra o link público deste experimento no Colab: [Abrir Perceptron OR](https://colab.research.google.com/github/Furriel/code-library/blob/main/cursos/ia/redes-neurais/perceptron-or.ipynb).
2. No menu **Ambiente de execução → Executar tudo**, rode as células de cima para baixo.
3. Faça a conta à mão antes de verificar a célula numérica.
4. Altere **somente o parâmetro sinalizado** no experimento guiado e execute tudo novamente.
5. Compare os gráficos e registre as conclusões; você não precisa enviar o notebook nem utilizar GitHub.

### Dados e parâmetros
O erro por época, a evolução dos pesos, a reta de decisão e o efeito de modificar apenas a taxa de aprendizagem.

O ambiente usa NumPy e Matplotlib disponíveis no Colab. O compilador C é opcional onde houver exemplo de referência.


In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

## 1. Perceptron: problema e equações

Dois indicadores binários de um motor — corrente elevada ($x_1$) e temperatura elevada ($x_2$) — devem produzir um alerta didático $d=1$ quando ao menos um estiver ativo (tabela OR). Em um equipamento real, essa regra seria programada diretamente; aqui ela serve para observar **como os coeficientes são aprendidos a partir de exemplos**.

$$z = w_1x_1 + w_2x_2 + b,\qquad y = \begin{cases}1, & z \ge 0\\ 0, & z<0\end{cases},\qquad e = d - y$$

$$w_i \leftarrow w_i + \eta\,e\,x_i,\qquad b \leftarrow b + \eta\,e\qquad(\text{apenas quando } e\neq 0)$$

A regra não deriva o degrau; ela não é retropropagação. Uma **época** percorre os quatro exemplos.

## 2. Cálculo manual e conferência

Com $x=(1,0)$, $w=(-0{,}2;\,0{,}1)$, $b=-0{,}1$, $d=1$, $\eta=0{,}2$: calcule $z$, $y$, $e$ e os novos parâmetros antes de executar a célula.

In [ ]:
x, w, b, d, eta = [1.0, 0.0], [-0.2, 0.1], -0.1, 1, 0.2
z = b
for i in range(2):
    z += w[i] * x[i]
y = int(z >= 0); e = d - y
w_novo = [w[i] + eta * e * x[i] for i in range(2)]
b_novo = b + eta * e
z_novo = b_novo + w_novo[0] * x[0] + w_novo[1] * x[1]
print(f"z = {z:.3f} -> y = {y}; e = {e}; w' = {w_novo}; b' = {b_novo:.3f}; z' = {z_novo:.3f} -> y' = {int(z_novo >= 0)}")
assert abs(z + 0.3) < 1e-12 and y == 0 and e == 1
assert abs(w_novo[0]) < 1e-12 and abs(w_novo[1] - 0.1) < 1e-12 and abs(b_novo - 0.1) < 1e-12
print("CÁLCULO MANUAL CONFERIDO: w2 não mudou porque x2 = 0.")

## 3. O perceptron em C (implementação de referência)

O programa abaixo é o mesmo `perceptron.c` do repositório. A parada ocorre na primeira época sem erros. Os quatro casos testados são os próprios exemplos de treino: **4/4 não demonstra generalização**.

In [ ]:
%%writefile perceptron.c
/*
 * perceptron.c — um neurônio artificial que APRENDE a tabela OR (C11, sem bibliotecas de IA)
 * ------------------------------------------------------------------------------------------
 * Projeto 1 — Inteligência Artificial aplicada à Engenharia Elétrica
 *
 * PROBLEMA: dois indicadores binários de um motor
 *     x1 = corrente acima de um limite didático   (0 ou 1)
 *     x2 = temperatura acima de um limite didático (0 ou 1)
 * devem gerar um alerta d = 1 quando ao menos um for 1 (função lógica OR).
 * Em vez de programar "if (x1 || x2)", deixamos o neurônio descobrir os pesos.
 *
 * EQUAÇÕES
 *     z = b + w1*x1 + w2*x2          soma ponderada (potencial)
 *     y = 1 se z >= 0, senão 0       ativação degrau
 *     erro = d - y                   pode ser -1, 0 ou +1
 *     wi <- wi + eta*erro*xi         regra do perceptron (só muda se errou)
 *     b  <- b  + eta*erro
 *
 * NÃO é um dispositivo de proteção elétrica: é um exercício sobre o mecanismo
 * de aprendizagem. A página da aula reproduz estas contas em JavaScript, na
 * mesma ordem, e um teste automático confere que os dois logs são iguais.
 *
 * Compilar: gcc -std=c11 -Wall -Wextra -pedantic -O2 perceptron.c -o perceptron
 * Executar: ./perceptron        (no Windows: .\perceptron.exe)
 */
#include <stdio.h>

#define N_ENTRADAS 2      /* x1 e x2 */
#define N_AMOSTRAS 4      /* as 4 linhas da tabela OR */
#define MAX_EPOCAS 100    /* limite de segurança: o treino nunca roda para sempre */

/* O "estado" do neurônio: tudo o que ele aprende cabe nestes três números. */
typedef struct {
    double pesos[N_ENTRADAS];   /* w1, w2 */
    double vies;                /* b: desloca a fronteira mesmo quando x = (0,0) */
} Perceptron;

/* z = b + soma(wi * xi). Começa pelo viés e soma produto a produto. */
static double potencial(const Perceptron *p, const double x[N_ENTRADAS]) {
    double z = p->vies;
    for (int i = 0; i < N_ENTRADAS; ++i) {
        z += p->pesos[i] * x[i];
    }
    return z;
}

/* Função degrau: transforma o número real z em uma classe (0 ou 1). */
static int degrau(double z) {
    return z >= 0.0 ? 1 : 0;
}

/* Previsão = degrau(potencial). É o que o neurônio "responde" para x. */
static int prever(const Perceptron *p, const double x[N_ENTRADAS]) {
    return degrau(potencial(p, x));
}

/*
 * Treinamento. Uma ÉPOCA percorre os 4 exemplos em ordem; para cada um:
 *   1. calcula a previsão y;
 *   2. calcula o erro d - y;
 *   3. se errou, corrige pesos e viés imediatamente (o próximo exemplo já
 *      usa os valores corrigidos — por isso a ordem dos exemplos importa).
 * Para quando uma época inteira termina sem erros e devolve o número da época;
 * devolve -1 se atingir max_epocas sem convergir.
 */
static int treinar(Perceptron *p,
                   const double entradas[N_AMOSTRAS][N_ENTRADAS],
                   const int desejado[N_AMOSTRAS],
                   double taxa, int max_epocas) {
    for (int epoca = 1; epoca <= max_epocas; ++epoca) {
        int erros = 0;
        for (int j = 0; j < N_AMOSTRAS; ++j) {
            const int y = prever(p, entradas[j]);
            const int diferenca = desejado[j] - y;          /* erro e = d - y */
            if (diferenca != 0) {
                ++erros;
                for (int i = 0; i < N_ENTRADAS; ++i) {
                    /* se a entrada xi for 0, este produto é 0 e o peso não muda */
                    p->pesos[i] += taxa * diferenca * entradas[j][i];
                }
                p->vies += taxa * diferenca;
            }
        }
        /* Log de uma linha por época: erros encontrados e parâmetros ao FINAL da época. */
        printf("epoca=%02d erros=%d w1=%+.3f w2=%+.3f b=%+.3f\n",
               epoca, erros, p->pesos[0], p->pesos[1], p->vies);
        if (erros == 0) return epoca;                       /* convergiu */
    }
    return -1;                                              /* não convergiu no limite */
}

int main(void) {
    /* Tabela OR: entradas (x1, x2) e saídas desejadas d. */
    const double x[N_AMOSTRAS][N_ENTRADAS] = {
        {0.0, 0.0}, {0.0, 1.0}, {1.0, 0.0}, {1.0, 1.0}
    };
    const int d[N_AMOSTRAS] = {0, 1, 1, 1};
    Perceptron p = {{0.0, 0.0}, 0.0};          /* começa com tudo zero */

    printf("Tabela OR: corrente_alta, temperatura_alta => alerta\n");
    const int epocas = treinar(&p, x, d, 0.2, MAX_EPOCAS);   /* taxa eta = 0,2 */
    if (epocas < 0) {
        fprintf(stderr, "Falha: nao convergiu no limite de epocas.\n");
        return 1;
    }
    printf("Convergiu em %d epocas\n", epocas);

    /* Teste: confere a previsão de cada exemplo. ATENÇÃO: são os mesmos
     * exemplos do treino, então 4/4 mostra que a tabela foi aprendida,
     * não que o modelo generaliza para casos novos. */
    for (int j = 0; j < N_AMOSTRAS; ++j) {
        const double z = potencial(&p, x[j]);
        const int y = prever(&p, x[j]);
        printf("teste=%d x1=%.0f x2=%.0f z=%+.3f previsto=%d esperado=%d\n",
               j, x[j][0], x[j][1], z, y, d[j]);
        if (y != d[j]) return 2;               /* código de saída diferente de 0 = falha */
    }
    printf("TESTES PASSARAM: 4/4\n");
    return 0;
}


In [ ]:
if GCC:
    comp = subprocess.run(["gcc", "-std=c11", "-Wall", "-Wextra", "-pedantic", "-O2", "perceptron.c", "-o", "perceptron"],
                          capture_output=True, text=True)
    print(comp.stdout + comp.stderr)
    assert comp.returncode == 0, "Erro de compilação"
    exec_c = subprocess.run(["./perceptron"], capture_output=True, text=True)
    print(exec_c.stdout)
    assert exec_c.returncode == 0 and "TESTES PASSARAM: 4/4" in exec_c.stdout
else:
    exec_c = None
    print("GCC ausente: execução do C pulada. O restante do notebook continua.")

## 4. O mesmo algoritmo em Python: histórico de pesos, erros e fronteira

A função `treinar_perceptron` repete as operações do C na mesma ordem. Ela registra os parâmetros **a cada passo** e os erros **a cada época**, e detecta ciclos (estado do fim de uma época repetido).

In [ ]:
ENTRADAS = [(0, 0), (0, 1), (1, 0), (1, 1)]
TABELAS = {"OR": [0, 1, 1, 1], "AND": [0, 0, 0, 1], "XOR": [0, 1, 1, 0]}

def treinar_perceptron(tabela, w0=(0.0, 0.0), b0=0.0, eta=0.2, max_epocas=100):
    w, b = list(w0), b0
    passos, epocas, vistos = [(w[0], w[1], b)], [], {}
    for ep in range(1, max_epocas + 1):
        erros = 0
        for x, d in zip(ENTRADAS, TABELAS[tabela]):
            z = b
            for i in range(2):
                z += w[i] * x[i]
            e = d - (1 if z >= 0 else 0)
            if e != 0:
                erros += 1
                for i in range(2):
                    w[i] += eta * e * x[i]
                b += eta * e
            passos.append((w[0], w[1], b))
        epocas.append((ep, erros, w[0], w[1], b))
        if erros == 0:
            return "convergiu", passos, epocas
        chave = tuple(round(v, 9) for v in (w[0], w[1], b))
        if chave in vistos:
            return f"ciclo (repete a época {vistos[chave]})", passos, epocas
        vistos[chave] = ep
    return "limite", passos, epocas

status_or, passos_or, epocas_or = treinar_perceptron("OR")
print("OR:", status_or, "em", len(epocas_or), "épocas")
for ep, er, w1, w2, bb in epocas_or:
    print(f"epoca={ep:02d} erros={er} w1={w1:+.3f} w2={w2:+.3f} b={bb:+.3f}")
if exec_c is not None:
    import re
    log_c = re.findall(r"epoca=(\d+) erros=(\d+) w1=([+-]\d+\.\d+) w2=([+-]\d+\.\d+) b=([+-]\d+\.\d+)", exec_c.stdout)
    log_py = [(f"{ep:02d}", str(er), f"{w1:+.3f}", f"{w2:+.3f}", f"{bb:+.3f}") for ep, er, w1, w2, bb in epocas_or]
    assert log_c == log_py, "Python e C divergiram"
    print("Python reproduz o log do C época a época.")

In [ ]:
passos = np.array(passos_or)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for k, (nome, cor) in enumerate([("w₁", "#2563a8"), ("w₂", "#0f7b74"), ("b", "#c2570c")]):
    ax[0].step(range(len(passos)), passos[:, k], where="post", label=nome, color=cor)
ax[0].set(title="Parâmetros a cada passo (OR, η = 0,2)", xlabel="passo de treinamento (1 exemplo)", ylabel="valor (adimensional)")
ax[0].legend()
ax[1].bar([e[0] for e in epocas_or], [e[1] for e in epocas_or], color="#b42318")
ax[1].set(title="Erros por época", xlabel="época", ylabel="exemplos classificados errado (de 4)", ylim=(0, 4))
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
for ep, er, w1, w2, bb in epocas_or:
    if abs(w2) > 1e-12:
        ax[2].plot(xx[0], -(w1 * xx[0] + bb) / w2, label=f"fim da época {ep}", alpha=0.4 + 0.6 * ep / len(epocas_or))
    elif abs(w1) > 1e-12:
        ax[2].axvline(-bb / w1, label=f"fim da época {ep}")
for (x1, x2), d in zip(ENTRADAS, TABELAS["OR"]):
    ax[2].scatter(x1, x2, s=160, c="#146e83" if d else "white", edgecolors="#18324b", zorder=3)
    ax[2].annotate(str(d), (x1 + 0.05, x2 + 0.05))
ax[2].set(xlim=(-0.3, 1.3), ylim=(-0.3, 1.3), title="Fronteira w₁x₁ + w₂x₂ + b = 0", xlabel="x₁: corrente elevada", ylabel="x₂: temperatura elevada")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
w1, w2, bb = epocas_or[-1][2:]
print("Classificação final:", [(x, int(bb + w1 * x[0] + w2 * x[1] >= 0)) for x in ENTRADAS])
print("Observe: (0,1) e (1,0) ficam exatamente sobre a reta (z = 0) e são classe 1 pela convenção z ≥ 0 — solução frágil.")

**Interpretação física.** A reta separa a região “sem alerta” da região “com alerta”. O viés desloca a reta; os pesos a giram. Como dois exemplos ficam exatamente sobre a fronteira, um pequeno ruído numa medição contínua mudaria a decisão — uma margem maior exigiria outro critério de treinamento.

## 5. Experimento guiado A — Perceptron: antes × depois

**Pergunta:** aumentar a taxa de aprendizagem faz o perceptron aprender OR mais rapidamente?

1. **Preveja:** antes de executar, anote se você espera menos, mais ou igual número de épocas ao mudar `ETA_ALTERADA` de 0,2 para 0,5.
2. **Execute:** a célula treina duas redes do zero, com os mesmos quatro exemplos, pesos iniciais e limite. Somente a taxa de aprendizagem muda.
3. **Observe:** a tabela, os erros por época e a interpretação calculada logo abaixo.
4. **Experimente novamente:** troque só `ETA_ALTERADA` por 0,1 ou 1,0 e execute esta célula de novo; a referência permanece em 0,2.
5. **Registre:** valor original → novo, épocas de treino, diferença observada e por que uma taxa maior não garante convergência em menos épocas.

O comentário é calculado pelos resultados reais desta execução; **não é resposta gerada por LLM**. Os quatro exemplos OR são didáticos; acertá-los não demonstra generalização para dados reais.

Inclua a comparação e sua explicação no **relatório final único do Moodle**.


In [ ]:
# ALTERE APENAS ESTE VALOR E EXECUTE NOVAMENTE ESTA CÉLULA.
ETA_ALTERADA = 0.5
ETA_ORIGINAL = 0.2
assert 0 < ETA_ALTERADA <= 1, "Use uma taxa entre 0 e 1."

s0, _, e0 = treinar_perceptron("OR", eta=ETA_ORIGINAL)
s1, _, e1 = treinar_perceptron("OR", eta=ETA_ALTERADA)
print("ANTES × DEPOIS — Perceptron OR")
print(f"Original: eta={ETA_ORIGINAL:g}; status={s0}; épocas={len(e0)}; erros finais={e0[-1][1]}")
print(f"Alterado: eta={ETA_ALTERADA:g}; status={s1}; épocas={len(e1)}; erros finais={e1[-1][1]}")
fig, ax = plt.subplots(figsize=(9, 4))
ax.step([z[0] for z in e0], [z[1] for z in e0], where="post", marker="o", label=f"original η={ETA_ORIGINAL:g}")
ax.step([z[0] for z in e1], [z[1] for z in e1], where="post", marker="s", label=f"alterado η={ETA_ALTERADA:g}")
ax.set(xlabel="época", ylabel="erros nos quatro padrões", title="Perceptron OR: comparação controlada")
ax.set_ylim(bottom=-.1)
ax.legend(); plt.tight_layout(); plt.show()
print("\nINTERPRETAÇÃO AUTOMÁTICA (baseada nos resultados observados):")
if s0 == "convergiu" and s1 == "convergiu":
    d_ep = len(e1) - len(e0)
    if d_ep < 0:
        print(f"A alteração precisou de {-d_ep} época(s) a menos neste exemplo.")
    elif d_ep > 0:
        print(f"A alteração precisou de {d_ep} época(s) a mais neste exemplo.")
    else:
        print("Ambas precisaram da mesma quantidade de épocas.")
    print("A taxa define o tamanho das atualizações; não garante redução de épocas em outros problemas.")
else:
    print("Pelo menos uma execução não convergiu no limite escolhido. Não conclua que a taxa maior é melhor.")
print("Para o relatório único no Moodle: compare as duas curvas e justifique o efeito observado.")



## Questões de interpretação

Compare o valor de z e os pesos calculados à mão com o código. Aumentar η sempre reduziu as épocas? Registre tabela-verdade, conta manual, curvas antes × depois e interpretação no relatório único do Moodle.

## Conclusão

Este experimento é independente. Use o mesmo conjunto de dados e as mesmas condições iniciais ao comparar parâmetros. **Experimentos propostos:** altere apenas o valor indicado no código e justifique o resultado observado, sem afirmar melhora quando as métricas não a demonstram. Entregue somente um **único relatório final** pelo Moodle, conforme as instruções da disciplina.
